In [ ]:
import polars as pl
import os
import tqdm
import logging

In [41]:
gtf_file = '/workspace-SR003.nfs2/estsoi/TSSprediction/GENA_LM/downstream_tasks/TSSprediction/data/annotations/GCA_014281875.1_ASM1428187v1_genomic.gtf'

In [42]:
os.path.relpath(gtf_file)

'data/annotations/GCA_014281875.1_ASM1428187v1_genomic.gtf'

In [ ]:
gtftools.get_tss_region(gtf_file, tss_bed_file='file.bed')

In [43]:
def locate_TSS_sites(GTFfile):

    TSSbed=[]

    f = open(GTFfile)
    for line in f:
        table = line.split('\t')
        if len(table) < 3:
            continue
        if table[2] == 'transcript':
            chrom  = table[0]
            strand = table[6] 
            tcx = line.split('transcript_id')[1].split('"')[1]
            geneid = line.split('gene_id')[1].split('"')[1]
            if "gene_name" in line:
                genesymbol = line.split('gene_name')[1].split('"')[1]
            else:
                genesymbol = geneid
            if  strand == "+":
                iregion = {'chrom':chrom, 'TSS':int(table[3])-1, 'strand':strand, 'geneid':geneid, 'gene_symbol':genesymbol, 'annotation': os.path.relpath(GTFfile)}
            elif strand == '-':
                iregion = {'chrom':chrom, 'TSS':int(table[4]), 'strand':strand, 'geneid':geneid, 'gene_symbol':genesymbol, 'annotation': os.path.relpath(GTFfile)}
            TSSbed.append(iregion)
    
    f.close()
    return pl.DataFrame(TSSbed)

In [91]:
tss = locate_TSS_sites(gtf_file)
metadata = pl.read_csv('metadata/condensed_metadata.csv').with_columns(annotation = pl.col('annotation').str.strip_suffix('.gz'))

In [92]:
a = tss.join(metadata, on='annotation', how='left')

In [94]:
a

chrom,TSS,strand,geneid,gene_symbol,annotation,genome_acc,genomePath,taxon
str,i64,str,str,str,str,str,str,str
"""CM024789.1""",7885,"""+""","""GBF38_003607""","""GBF38_003607""","""data/annotations/GCA_014281875…","""GCA_014281875.1""","""data/genomes/Nibea_albiflora.f…","""Actinopteri"""
"""CM024789.1""",14296,"""-""","""GBF38_003608""","""GBF38_003608""","""data/annotations/GCA_014281875…","""GCA_014281875.1""","""data/genomes/Nibea_albiflora.f…","""Actinopteri"""
"""CM024789.1""",15664,"""-""","""GBF38_003609""","""GBF38_003609""","""data/annotations/GCA_014281875…","""GCA_014281875.1""","""data/genomes/Nibea_albiflora.f…","""Actinopteri"""
"""CM024789.1""",23489,"""-""","""GBF38_003610""","""GBF38_003610""","""data/annotations/GCA_014281875…","""GCA_014281875.1""","""data/genomes/Nibea_albiflora.f…","""Actinopteri"""
"""CM024789.1""",30621,"""-""","""GBF38_003611""","""GBF38_003611""","""data/annotations/GCA_014281875…","""GCA_014281875.1""","""data/genomes/Nibea_albiflora.f…","""Actinopteri"""
…,…,…,…,…,…,…,…,…
"""JABGLX010000371.1""",9189,"""-""","""GBF38_000009""","""GBF38_000009""","""data/annotations/GCA_014281875…","""GCA_014281875.1""","""data/genomes/Nibea_albiflora.f…","""Actinopteri"""
"""JABGLX010000371.1""",11291,"""-""","""GBF38_000010""","""GBF38_000010""","""data/annotations/GCA_014281875…","""GCA_014281875.1""","""data/genomes/Nibea_albiflora.f…","""Actinopteri"""
"""JABGLX010000371.1""",22124,"""-""","""GBF38_000008""","""GBF38_000008""","""data/annotations/GCA_014281875…","""GCA_014281875.1""","""data/genomes/Nibea_albiflora.f…","""Actinopteri"""


In [93]:
a.schema

Schema([('chrom', String),
        ('TSS', Int64),
        ('strand', String),
        ('geneid', String),
        ('gene_symbol', String),
        ('annotation', String),
        ('genome_acc', String),
        ('genomePath', String),
        ('taxon', String)])

In [85]:
pl.Schema([('chrom', pl.String),
        ('TSS', pl.Int64),
        ('strand', pl.String),
        ('geneid', pl.String),
        ('gene_symbol', pl.String),
        ('annotation', pl.String),
        ('genome_acc', pl.String),
        ('genomePath', pl.String),
        ('taxon', pl.String)])

Schema([('chrom', String),
        ('TSS', Int64),
        ('strand', String),
        ('geneid', String),
        ('gene_symbol', String),
        ('annotation', String),
        ('genome_acc', String),
        ('genomePath', String),
        ('taxon', String)])

In [48]:
annotationsDir = 'data/annotations'

In [86]:
final_mapping = pl.DataFrame(schema=pl.Schema([('chrom', pl.String),
        ('TSS', pl.Int64),
        ('strand', pl.String),
        ('geneid', pl.String),
        ('gene_symbol', pl.String),
        ('annotation', pl.String),
        ('genome_acc', pl.String),
        ('genomePath', pl.String),
        ('taxon', pl.String)]))

In [101]:
metadata = pl.read_csv('metadata/condensed_metadata.csv').with_columns(annotation = pl.col('annotation').str.strip_suffix('.gz'))
annotations = os.listdir(annotationsDir)
for annotation in tqdm.tqdm(annotations, total = len(annotations)):
    if 'gtf' not in annotation:
        continue
    annotation_full_path = os.path.join(annotationsDir, annotation)
    tss_df = locate_TSS_sites(annotation_full_path)
    assert len(tss_df) > 0
    tss_df = tss_df.join(metadata, on='annotation', how='left')
    final_mapping = pl.concat([final_mapping, tss_df])
    

100%|██████████| 70/70 [02:33<00:00,  2.20s/it]


In [103]:
final_mapping.write_csv('dataset.csv')

In [2]:
final_mapping

NameError: name 'final_mapping' is not defined

In [110]:
final_mapping['taxon'].value_counts().sort('count')

taxon,count
str,u32
"""Chondrichthyes""",40718
"""Myxini""",48017
"""Lepidosauria""",126881
"""Amphibia""",373271
"""Aves""",431252
"""Actinopteri""",1541877
"""Mammalia""",2937895


In [5]:
dataset = pl.read_csv('/workspace-SR003.nfs2/estsoi/TSSprediction/GENA_LM/downstream_tasks/TSSprediction/dataset.csv')

In [8]:
dataset.with_columns(genome_path = pl.col('genome_path').str.strip_suffix('.gz')).write_csv('/workspace-SR003.nfs2/estsoi/TSSprediction/GENA_LM/downstream_tasks/TSSprediction/dataset.csv')

In [8]:
dataset.with_columns(is_TSS=1).write_csv('/workspace-SR003.nfs2/estsoi/TSSprediction/GENA_LM/downstream_tasks/TSSprediction/dataset.csv')

In [ ]:
from TSS_dataset import TSSDataset

/home/jovyan/miniconda3/envs/exprflash/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
TSSDataset(mapping_file='/workspace-SR003.nfs2/estsoi/TSSprediction/GENA_LM/downstream_tasks/TSSprediction/dataset.csv',
            tokenizer='/workspace-SR003.nfs2/estsoi/TSSprediction/GENA_LM/data/tokenizers/t2t_1000h_multi_32k',
            cache_dir='/workspace-SR003.nfs2/estsoi/TSSprediction/GENA_LM/downstream_tasks/TSSprediction/cache',
            num_before = 512,
            token_len_for_fetch = 10,
            max_seq_len = 1024,
            loglevel = logging.WARNING,
            data_workers = 10)

Tokenizing:   5%|▍         | 264817/5499911 [27:43<9:29:05, 153.32it/s] 

KeyboardInterrupt: 